In [1]:
# CELL 1 — Phase 6 config and dataset loading
import numpy as np
import torch
import torch.nn as nn
import math
import time
import json
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, matthews_corrcoef
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from scipy.stats import combine_pvalues

N_QUBITS = 4
ENTANGLING_LAYERS = 2
D_MODEL = 64
D_FF = 128
N_HEADS = 4
N_TOKENS = 225
PATCH_SIDE = 15
TRAIN_BATCH_SIZE = 32
EPOCHS = 50
LR = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SEEDS = [42, 43, 44]
NOISE_STD = 0.02

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

d = np.load("preprocessed/Houston2013_Full15.npz")
train_tokens = torch.tensor(d["train_tokens"], dtype=torch.float32)
train_labels = torch.tensor(d["train_labels"] - 1, dtype=torch.long)
val_tokens = torch.tensor(d["val_tokens"], dtype=torch.float32)
val_labels = torch.tensor(d["val_labels"] - 1, dtype=torch.long)
test_tokens = torch.tensor(d["test_tokens"], dtype=torch.float32)
test_labels = torch.tensor(d["test_labels"] - 1, dtype=torch.long)

K_DIM = train_tokens.shape[-1]
N_CLASSES = int(train_labels.max().item()) + 1
print(f"Houston2013_Full15: k={K_DIM}, classes={N_CLASSES}")
print(f"train={train_tokens.shape}, val={val_tokens.shape}, test={test_tokens.shape}")

Using device: cuda
Houston2013_Full15: k=10, classes=15
train=torch.Size([2535, 225, 10]), val=torch.Size([282, 225, 10]), test=torch.Size([12182, 225, 10])


In [2]:
# CELL 2 — All ablation encoder variants + augmentation (reused from Phase 4/5)
import pennylane as qml

QUANTUM_DEVICE_NAME = "default.qubit"
DIFF_METHOD = "backprop"

def build_quantum_layer(n_qubits):
    dev = qml.device(QUANTUM_DEVICE_NAME, wires=n_qubits)
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=ENTANGLING_LAYERS, n_wires=n_qubits)
    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]
    return qml.qnn.TorchLayer(circuit, {"weights": weight_shape})

class FullQuantumEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, N_QUBITS)
        self.q_layer = build_quantum_layer(N_QUBITS)
        self.out_proj = nn.Linear(N_QUBITS, D_MODEL)
    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, N_QUBITS)
        q_out = self.q_layer(flat).reshape(b, n, N_QUBITS)
        return self.out_proj(q_out)

class OneQubitEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, 1)
        self.q_layer = build_quantum_layer(1)
        self.out_proj = nn.Linear(1, D_MODEL)
    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, 1)
        q_out = self.q_layer(flat).reshape(b, n, 1)
        return self.out_proj(q_out)

class ClassicalMirrorEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(k_dim, N_QUBITS), nn.Tanh(), nn.Linear(N_QUBITS, D_MODEL))
    def forward(self, tokens):
        return self.net(tokens)

class FullRankEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.net = nn.Linear(k_dim, D_MODEL)
    def forward(self, tokens):
        return self.net(tokens)

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_tokens, d_model):
        super().__init__()
        pe = torch.zeros(n_tokens, d_model)
        pos = torch.arange(0, n_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe

ENCODER_REGISTRY = {
    "Full": FullQuantumEncoder, "w/o QNN": ClassicalMirrorEncoder,
    "w/o Bottleneck": FullRankEncoder, "1-Qubit": OneQubitEncoder,
    "w/o PosEnc": FullQuantumEncoder,
}

class AblationQuantFormer(nn.Module):
    def __init__(self, k_dim, n_classes, config_name):
        super().__init__()
        self.encoder_module = ENCODER_REGISTRY[config_name](k_dim)
        self.use_pos_enc = (config_name != "w/o PosEnc")
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.transformer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True, dropout=0.0)
        self.classifier = nn.Linear(D_MODEL, n_classes)
    def forward(self, tokens):
        x = self.encoder_module(tokens)
        if self.use_pos_enc:
            x = self.pos_enc(x)
        x = self.transformer(x)
        return self.classifier(x.mean(dim=1))

def augment_batch(tokens_flat, k_dim, rng, noise_std=NOISE_STD):
    n = tokens_flat.shape[0]
    grid = tokens_flat.reshape(n, PATCH_SIDE, PATCH_SIDE, k_dim).clone()
    for i in range(n):
        if rng.random() < 0.5:
            grid[i] = torch.flip(grid[i], dims=[0])
        if rng.random() < 0.5:
            grid[i] = torch.flip(grid[i], dims=[1])
        k_rot = rng.integers(0, 4)
        if k_rot > 0:
            grid[i] = torch.rot90(grid[i], k=k_rot, dims=[0, 1])
    grid = grid + torch.randn_like(grid) * noise_std  # documented deviation, per Phase 4
    return grid.reshape(n, PATCH_SIDE * PATCH_SIDE, k_dim)

print("Cell 2 loaded: 5 ablation configs + augmentation ready.")

Cell 2 loaded: 5 ablation configs + augmentation ready.


In [3]:
# CELL 3 — Training loop + evaluation
def get_param_groups(model):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        (no_decay if "q_layer" in name else decay).append(param)
    return [{"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0}]

def train_config(seed, k_dim, n_classes, config_name, train_tokens, train_labels,
                  val_tokens, val_labels, augment=False, aug_seed=None):
    torch.manual_seed(seed)
    model = AblationQuantFormer(k_dim, n_classes, config_name).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    best_val_acc, best_state = -1, None
    rng = np.random.default_rng(aug_seed) if augment else None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        xb_epoch = augment_batch(train_tokens, k_dim, rng) if augment else train_tokens
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = xb_epoch[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(val_labels, model(val_tokens.to(DEVICE)).argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

def evaluate_with_predictions(model, test_tokens, test_labels, n_classes):
    model.eval()
    with torch.no_grad():
        preds = model(test_tokens.to(DEVICE)).argmax(dim=1).cpu().numpy()
    labels_np = test_labels.numpy()
    cm = confusion_matrix(labels_np, preds, labels=list(range(n_classes)))
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    metrics = {"OA": accuracy_score(labels_np, preds), "AA": per_class_acc.mean(),
               "kappa": cohen_kappa_score(labels_np, preds), "MCC": matthews_corrcoef(labels_np, preds),
               "confusion_matrix": cm.tolist()}
    return metrics, preds

print("Cell 3 loaded.")

Cell 3 loaded.


In [4]:
# CELL 4 — Houston-15: Full x2 augmentation conditions, + 4 ablation configs unaugmented
houston_results = {}
houston_predictions = {}

# Full config: both augmentation conditions
houston_results["Full"] = {"unaugmented": [], "augmented": []}
houston_predictions["Full"] = {"unaugmented": {}, "augmented": {}}
for augment in [False, True]:
    tag = "augmented" if augment else "unaugmented"
    for seed in SEEDS:
        print(f"\n=== Houston15 [Full, {tag}], seed={seed} ===")
        start = time.time()
        model = train_config(seed, K_DIM, N_CLASSES, "Full", train_tokens, train_labels,
                              val_tokens, val_labels, augment=augment, aug_seed=seed + 1000)
        elapsed = time.time() - start
        metrics, preds = evaluate_with_predictions(model, test_tokens, test_labels, N_CLASSES)
        metrics["seed"], metrics["train_time_sec"] = seed, elapsed
        houston_results["Full"][tag].append(metrics)
        houston_predictions["Full"][tag][seed] = preds.tolist()
        print(f"  OA={metrics['OA']:.4f} AA={metrics['AA']:.4f} kappa={metrics['kappa']:.4f} ({elapsed/60:.1f} min)")

# Ablation configs: unaugmented only, per locked protocol
for config in ["w/o QNN", "w/o Bottleneck", "w/o PosEnc", "1-Qubit"]:
    houston_results[config] = []
    houston_predictions[config] = {}
    for seed in SEEDS:
        print(f"\n=== Houston15 [{config}], seed={seed} ===")
        start = time.time()
        model = train_config(seed, K_DIM, N_CLASSES, config, train_tokens, train_labels,
                              val_tokens, val_labels, augment=False)
        elapsed = time.time() - start
        metrics, preds = evaluate_with_predictions(model, test_tokens, test_labels, N_CLASSES)
        metrics["seed"], metrics["train_time_sec"] = seed, elapsed
        houston_results[config].append(metrics)
        houston_predictions[config][seed] = preds.tolist()
        print(f"  OA={metrics['OA']:.4f} AA={metrics['AA']:.4f} kappa={metrics['kappa']:.4f} ({elapsed/60:.1f} min)")

with open("phase6_houston15_results.json", "w") as f:
    json.dump(houston_results, f, indent=2)
with open("phase6_houston15_predictions.json", "w") as f:
    json.dump(houston_predictions, f, indent=2)
print("\nSaved phase6_houston15_results.json and phase6_houston15_predictions.json")


=== Houston15 [Full, unaugmented], seed=42 ===
  OA=0.7420 AA=0.7566 kappa=0.7203 (3.6 min)

=== Houston15 [Full, unaugmented], seed=43 ===
  OA=0.7480 AA=0.7649 kappa=0.7269 (3.6 min)

=== Houston15 [Full, unaugmented], seed=44 ===
  OA=0.7357 AA=0.7436 kappa=0.7136 (3.6 min)

=== Houston15 [Full, augmented], seed=42 ===
  OA=0.7791 AA=0.8003 kappa=0.7607 (3.9 min)

=== Houston15 [Full, augmented], seed=43 ===
  OA=0.7780 AA=0.8066 kappa=0.7610 (3.9 min)

=== Houston15 [Full, augmented], seed=44 ===
  OA=0.7608 AA=0.7844 kappa=0.7409 (3.9 min)

=== Houston15 [w/o QNN], seed=42 ===
  OA=0.6873 AA=0.6911 kappa=0.6605 (0.3 min)

=== Houston15 [w/o QNN], seed=43 ===
  OA=0.6956 AA=0.7109 kappa=0.6703 (0.3 min)

=== Houston15 [w/o QNN], seed=44 ===
  OA=0.6978 AA=0.7108 kappa=0.6723 (0.3 min)

=== Houston15 [w/o Bottleneck], seed=42 ===
  OA=0.7155 AA=0.7325 kappa=0.6914 (0.3 min)

=== Houston15 [w/o Bottleneck], seed=43 ===
  OA=0.7680 AA=0.7821 kappa=0.7482 (0.3 min)

=== Houston15 [w/o

In [5]:
# CELL 5 — McNemar's test for Houston-15 (baseline = Full, unaugmented)
MIN_PVALUE = 1e-300
comparison_configs = ["w/o QNN", "w/o Bottleneck", "w/o PosEnc", "1-Qubit"]
true_labels = test_labels.numpy()

houston_stats = {}
for comp_config in comparison_configs:
    per_seed_pvalues = []
    for seed in SEEDS:
        full_preds = np.array(houston_predictions["Full"]["unaugmented"][seed])
        comp_preds = np.array(houston_predictions[comp_config][seed])
        full_correct, comp_correct = (full_preds == true_labels), (comp_preds == true_labels)
        table = [[np.sum(full_correct & comp_correct), np.sum(full_correct & ~comp_correct)],
                 [np.sum(~full_correct & comp_correct), np.sum(~full_correct & ~comp_correct)]]
        result = mcnemar(table, exact=False, correction=True)
        per_seed_pvalues.append(max(result.pvalue, MIN_PVALUE))
    houston_stats[comp_config] = {"per_seed_pvalues": per_seed_pvalues}

for seed_idx in range(len(SEEDS)):
    pvals = [houston_stats[c]["per_seed_pvalues"][seed_idx] for c in comparison_configs]
    _, corrected, _, _ = multipletests(pvals, method="fdr_bh")
    corrected = [max(p, MIN_PVALUE) for p in corrected]
    for c, p in zip(comparison_configs, corrected):
        houston_stats[c].setdefault("bh_corrected_pvalues", []).append(p)

print(f"\n{'='*70}\nHOUSTON-15 McNEMAR SUMMARY (Full vs. each config)\n{'='*70}")
oa_full = np.mean([r["OA"] for r in houston_results["Full"]["unaugmented"]])
for comp_config in comparison_configs:
    _, combined_p = combine_pvalues(houston_stats[comp_config]["bh_corrected_pvalues"], method="fisher")
    houston_stats[comp_config]["combined_pvalue"] = combined_p
    houston_stats[comp_config]["significant"] = bool(combined_p < 0.05)
    oa_comp = np.mean([r["OA"] for r in houston_results[comp_config]])
    effect = (oa_comp - oa_full) * 100
    magnitude = "negligible" if abs(effect) < 0.5 else ("moderate" if abs(effect) < 3 else "large")
    print(f"  Full ({oa_full:.4f}) vs {comp_config} ({oa_comp:.4f}): effect={effect:+.2f}pp "
          f"[{magnitude}] combined p={combined_p:.4g} "
          f"{'[SIGNIFICANT]' if houston_stats[comp_config]['significant'] else '[not significant]'}")

aug_oa = np.mean([r["OA"] for r in houston_results["Full"]["augmented"]])
print(f"\nAugmentation effect on Full config: unaugmented={oa_full:.4f}, augmented={aug_oa:.4f}, "
      f"diff={(aug_oa - oa_full)*100:+.2f}pp")

with open("phase6_mcnemar_results.json", "w") as f:
    json.dump(houston_stats, f, indent=2)
print("\nSaved phase6_mcnemar_results.json")


HOUSTON-15 McNEMAR SUMMARY (Full vs. each config)
  Full (0.7419) vs w/o QNN (0.6936): effect=-4.83pp [large] combined p=9.178e-96 [SIGNIFICANT]
  Full (0.7419) vs w/o Bottleneck (0.7356): effect=-0.63pp [moderate] combined p=8.79e-19 [SIGNIFICANT]
  Full (0.7419) vs w/o PosEnc (0.7297): effect=-1.22pp [moderate] combined p=1.79e-08 [SIGNIFICANT]
  Full (0.7419) vs 1-Qubit (0.5710): effect=-17.09pp [large] combined p=0 [SIGNIFICANT]

Augmentation effect on Full config: unaugmented=0.7419, augmented=0.7726, diff=+3.08pp

Saved phase6_mcnemar_results.json
